# Drishti — VizWiz Baseline (stock model, no fine-tuning)

**Goal:** the Sem-7 baseline number. Run the stock VLM on N VizWiz-val questions and score
with the **official VizWiz accuracy metric**: `acc = mean( min(#matching-humans / 3, 1) )`
over the 10 crowd answers per question.

Everything we do later (LoRA fine-tuning on VizWiz + our Indian dataset) must beat this row:

| Model | N | Overall acc | Answerable acc | Unanswerable acc | s/answer |
|---|---|---|---|---|---|
| stock SmolVLM-Instruct | 500 | ? | ? | ? | ? |

**Base model: SmolVLM, not Moondream-2.** The notebook-00 spike measured both on real
VizWiz photos and SmolVLM won on all three axes that matter here:

| | Moondream-2 | SmolVLM |
|---|---|---|
| Latency | ~4.4 s/answer | **~1.75 s** (2.5× faster) |
| Answer style | verbose paragraphs | **terse** |
| Loading | `trust_remote_code` — breaks on transformers v5 | **native transformers classes** |

Terseness is not cosmetic: VizWiz scores by **exact match** against short crowd answers, so
a verbose-but-correct answer scores ~0. Moondream also fabricated a drug name and ingredient
list when asked about a medicine — the finding that motivates the `app/drug_db.py` guardrail.

Moondream remains selectable below for comparison, but it requires `transformers<5`.

Colab: `Runtime → T4 GPU → Run all` (~15 min for N=500 with SmolVLM).

In [4]:
%pip install -q -U transformers accelerate datasets einops
import torch, time, json, re, string, transformers
from itertools import islice
from datasets import load_dataset

print('transformers:', transformers.__version__)

# --- model selection -----------------------------------------------------------------
# SmolVLM uses native transformers classes, so it is immune to the trust_remote_code
# breakage that killed Moondream on transformers v5. It was also 2.5x faster and terser
# in the notebook-00 spike -- see this notebook's header for the comparison.
MODEL_ID = 'HuggingFaceTB/SmolVLM-Instruct'
# MODEL_ID = 'vikhyatk/moondream2'   # comparison only; REQUIRES pip install "transformers<5"

# Moondream needs transformers 4.x; fail fast with an actionable message instead of an
# obscure AttributeError two cells later.
if 'moondream' in MODEL_ID.lower() and int(transformers.__version__.split('.')[0]) >= 5:
    raise RuntimeError(
        f"Moondream needs transformers<5 but {transformers.__version__} is loaded.\n"
        f"Change the install line above to: %pip install -q 'transformers<5' ...\n"
        f"then Runtime -> Restart session -> Run all (a pip downgrade cannot replace an\n"
        f"already-imported module, which is why restarting is required)."
    )

N_SAMPLES = 500          # increase to full val (4319) for the report if time allows
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '| model:', MODEL_ID)

# VizWiz-specific prompt: the metric rewards saying 'unanswerable' when the photo is unusable
PROMPT_SUFFIX = (" Answer in one to three words. If the question cannot be answered"
                 " from the image, answer exactly: unanswerable")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 18.1 MB/s eta 0:00:00
transformers: 5.14.1
device: cuda | model: HuggingFaceTB/SmolVLM-Instruct


In [5]:
stream = load_dataset('lmms-lab/VizWiz-VQA', split='val', streaming=True)
data = list(islice(stream, N_SAMPLES))

def gt_answers(sample):
    """Normalize: answers may be list[str] or list[{'answer': ...}]."""
    ans = sample['answers']
    return [a['answer'] if isinstance(a, dict) else a for a in ans]

print(len(data), 'samples ·', 'fields:', list(data[0].keys()))
print('example answers:', gt_answers(data[0])[:4])

README.md:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

500 samples · fields: ['question_id', 'image', 'question', 'answers', 'category']
example answers: ['unanswerable', 'unanswerable', 'unanswerable', 'unanswerable']


In [6]:
# Loader dispatches on model family so the rest of the notebook is model-agnostic.
# SmolVLM -> native transformers classes. Moondream -> trust_remote_code (transformers 4.x).

if 'moondream' in MODEL_ID.lower():
    from transformers import AutoModelForCausalLM, AutoTokenizer

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, trust_remote_code=True, torch_dtype=torch.float16, device_map=DEVICE)
    _tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

    def answer(img, question):
        prompt = question + PROMPT_SUFFIX
        try:
            return model.query(img, prompt)['answer']
        except AttributeError:
            return model.answer_question(model.encode_image(img), prompt, _tok)

else:
    from transformers import AutoProcessor
    try:  # AutoModelForVision2Seq is deprecated in favour of this in transformers 5
        from transformers import AutoModelForImageTextToText as _VisionSeq
    except ImportError:
        from transformers import AutoModelForVision2Seq as _VisionSeq

    processor = AutoProcessor.from_pretrained(MODEL_ID)
    model = _VisionSeq.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map=DEVICE)
    model.eval()

    def answer(img, question):
        msgs = [{'role': 'user',
                 'content': [{'type': 'image'},
                             {'type': 'text', 'text': question + PROMPT_SUFFIX}]}]
        prompt = processor.apply_chat_template(msgs, add_generation_prompt=True)
        inputs = processor(text=prompt, images=[img.convert('RGB')], return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=24, do_sample=False)
        text = processor.batch_decode(out, skip_special_tokens=True)[0]
        return text.split('Assistant:')[-1].strip()

# smoke-test on one sample before committing to the full 500-question run
_s = data[0]
_t0 = time.time()
print('Q :', _s['question'])
print('A :', answer(_s['image'], _s['question']), f'({time.time() - _t0:.1f}s)')
print('GT:', gt_answers(_s)[:4])

processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/7.45k [00:00<?, ?B/s]

[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 128002. This may result in unexpected behavior.


...

model.safetensors: reconstructing file:   0%|          |  0.00B / 4.49GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/657 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

Q : Ok. There is another picture I hope it is a better one.
A : Camera (3.2s)
GT: ['unanswerable', 'unanswerable', 'unanswerable', 'unanswerable']


In [7]:
from tqdm.auto import tqdm

results = []
for s in tqdm(data):
    t0 = time.time()
    try:
        pred = answer(s['image'], s['question'])
    except Exception as e:
        pred = f'__error__ {e}'
    results.append({'question': s['question'], 'prediction': pred,
                    'answers': gt_answers(s), 'latency_s': round(time.time() - t0, 2)})
print('done:', len(results))

  0%|          | 0/500 [00:00<?, ?it/s]

done: 500


In [8]:
# Official-style VQA answer normalization (simplified) + VizWiz accuracy
ARTICLES = {'a', 'an', 'the'}
def norm(t):
    t = t.lower().strip()
    t = t.translate(str.maketrans('', '', string.punctuation))
    return ' '.join(w for w in t.split() if w not in ARTICLES)

def vizwiz_acc(pred, answers):
    p = norm(pred)
    matches = sum(norm(a) == p for a in answers)
    return min(matches / 3.0, 1.0)

for r in results:
    r['acc'] = vizwiz_acc(r['prediction'], r['answers'])
    r['unanswerable_gt'] = sum(norm(a) == 'unanswerable' for a in r['answers']) >= 5

overall = sum(r['acc'] for r in results) / len(results)
ans_set = [r for r in results if not r['unanswerable_gt']]
una_set = [r for r in results if r['unanswerable_gt']]
lat = sum(r['latency_s'] for r in results) / len(results)

print(f'BASELINE — {MODEL_ID} on {len(results)} VizWiz-val samples')
print(f'  overall accuracy      : {overall:.3f}')
print(f'  answerable subset     : {sum(r["acc"] for r in ans_set)/max(len(ans_set),1):.3f}  (n={len(ans_set)})')
print(f'  unanswerable subset   : {sum(r["acc"] for r in una_set)/max(len(una_set),1):.3f}  (n={len(una_set)})')
print(f'  mean latency (T4 GPU) : {lat:.2f}s/answer')

BASELINE — HuggingFaceTB/SmolVLM-Instruct on 500 VizWiz-val samples
  overall accuracy      : 0.308
  answerable subset     : 0.310  (n=256)
  unanswerable subset   : 0.306  (n=244)
  mean latency (T4 GPU) : 1.21s/answer


In [9]:
# Save results + inspect worst failures (these guide fine-tuning and prompt fixes)
import pandas as pd
df = pd.DataFrame([{k: v for k, v in r.items() if k != 'answers'} |
                   {'gt_sample': '; '.join(r['answers'][:3])} for r in results])
df.to_csv('vizwiz_baseline_results.csv', index=False)
print('saved vizwiz_baseline_results.csv — download and commit to eval/results/')
df[df.acc == 0].head(15)[['question', 'prediction', 'gt_sample']]

saved vizwiz_baseline_results.csv — download and commit to eval/results/


,question,prediction,gt_sample
0,Ok. There is another picture I hope it is a be...,Camera,unanswerable; unanswerable; unanswerable
1,Can you tell me what this medicine is please?,Honey.,no; unanswerable; night time
2,What is the title of this book?,Twelve years,dog years; dog years: memoir; dog years
3,Which one is the blue one?,Blue,right; right; right
4,What does the arrow say?,OK,unanswerable; unanswerable; zoomtext
5,What the screen says? Thank you.,Windows,unanswerable; unanswerable; dell
6,What is seen on the laptop screen? What is see...,Nothing.,unanswerable; unanswerable; unanswerable
8,Is this shampoo or conditioner?,Shampoo,unanswerable; unanswerable; unanswerable
9,Can you describe for me what's going on outsid...,Trees.,store front; unanswerable; nothing going outsi...
10,What about the signery.,Building.,unanswerable; unanswerable; political


## Record for the report

1. Fill the baseline table row (top of notebook) with these numbers.
2. Note 3 failure patterns from the worst-failures table (e.g. verbose answers, missed
   'unanswerable', OCR-type questions) — these motivate the fine-tuning + routing design.
3. Repeat later with the LoRA fine-tuned model — same N, same prompt — for a fair comparison.